In [0]:
# Imports Block

import requests
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from delta.tables import *
import time
from datetime import datetime




In [ ]:
dbutils.widgets.removeAll()
# Widgets para pametrizar
dbutils.widgets.text('pokemon_limit', '151', 'Pokemon numbers')
dbutils.widgets.text('ingestion_date', datetime.now().strftime('%Y-%m-%d'), 'Partition date')

#Paths and configurarions
bronze_table = 'pokemons.bronze.pokemons_raw'

pokemon_limit = int(dbutils.widgets.get('pokemon_limit'))
ingestion_ts = datetime.now()
ingestion_date = dbutils.widgets.get('ingestion_date')

# Fail fast on a malformed date: it is used in replaceWhere and in partition filters.
datetime.strptime(ingestion_date, '%Y-%m-%d')
url = 'https://pokeapi.co/api/v2/pokemon'

print(f'DEBUG: pokemon_limit={pokemon_limit}')
print(f'DEBUG: ingestion_ts={ingestion_ts}')
print(f'DEBUG: ingestion_date={ingestion_date}')
print(f'DEBUG: url={url}')

In [0]:
# Raw Schema
raw_schema = StructType([
    StructField('pokemon_id', IntegerType(), False),
    StructField('raw_json', StringType(), True),
    StructField('ingestion_ts', TimestampType(), False),
    StructField('ingestion_date', StringType(), False),
    StructField('source_system', StringType(), False),
    StructField('api_status_code', IntegerType(), True)
])

print(raw_schema, type(raw_schema))

StructType([StructField('pokemon_id', IntegerType(), False), StructField('raw_json', StringType(), True), StructField('ingestion_ts', TimestampType(), False), StructField('ingestion_date', StringType(), False), StructField('source_system', StringType(), False), StructField('api_status_code', IntegerType(), True)]) <class 'pyspark.sql.types.StructType'>


In [ ]:
pokemon_data = []
for pid in range(1, pokemon_limit + 1):
    try:
        print(f'{url}/{pid}')
        response = requests.get(f'{url}/{pid}', timeout=10)
        pokemon_data.append({
            'pokemon_id': pid,
            'raw_json': response.text,
            'ingestion_ts': ingestion_ts,
            'ingestion_date': ingestion_date,
            'source_system': 'pokeapi_co',
            'api_status_code': response.status_code
        })
        time.sleep(0.1)
    except requests.RequestException as error:
        # No response exists when the request itself fails: keep the row, with no payload or status.
        print(f'ERROR: {url}/{pid} failed: {error}')
        pokemon_data.append({
            'pokemon_id': pid,
            'raw_json': None,
            'ingestion_ts': ingestion_ts,
            'ingestion_date': ingestion_date,
            'source_system': 'pokeapi_co',
            'api_status_code': None
        })

In [0]:
df_raw = spark.createDataFrame(pokemon_data, schema=raw_schema)

df_raw = df_raw.filter(
    F.col('pokemon_id').isNotNull() &
    F.col('ingestion_ts').isNotNull()
)

print(df_raw)

DataFrame[pokemon_id: int, raw_json: string, ingestion_ts: timestamp, ingestion_date: string, source_system: string, api_status_code: int]


In [0]:
df_raw.printSchema()
df_raw.show(3)

root
 |-- pokemon_id: integer (nullable = false)
 |-- raw_json: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = false)
 |-- ingestion_date: string (nullable = false)
 |-- source_system: string (nullable = false)
 |-- api_status_code: integer (nullable = true)

+----------+--------------------+--------------------+--------------+-------------+---------------+
|pokemon_id|            raw_json|        ingestion_ts|ingestion_date|source_system|api_status_code|
+----------+--------------------+--------------------+--------------+-------------+---------------+
|         1|{"abilities":[{"a...|2026-03-20 00:45:...|    2026-03-19|   pokeapi_co|            200|
|         2|{"abilities":[{"a...|2026-03-20 00:45:...|    2026-03-19|   pokeapi_co|            200|
|         3|{"abilities":[{"a...|2026-03-20 00:45:...|    2026-03-19|   pokeapi_co|            200|
+----------+--------------------+--------------------+--------------+-------------+---------------+
only showing top 3 ro

In [ ]:
# Idempotent load: rerunning a date replaces only that partition instead of appending duplicates.
(df_raw
 .write
 .format('delta')
 .mode('overwrite')
 .option('replaceWhere', f"ingestion_date = '{ingestion_date}'")
 .partitionBy('ingestion_date')
 .saveAsTable(bronze_table)
 )

In [0]:
spark.sql(f'SELECT * FROM {bronze_table}').show(3)

spark.sql(f'DESCRIBE EXTENDED {bronze_table}').display()

spark.sql(f"""
    SELECT 
        ingestion_date,
        COUNT(*) as total_records,
        COUNT(raw_json) as valid_json,
        SUM(CASE WHEN api_status_code = 200 THEN 1 ELSE 0 END) as success_api
    FROM {bronze_table}
    GROUP BY ingestion_date
""").display()


+----------+--------------------+--------------------+--------------+-------------+---------------+
|pokemon_id|            raw_json|        ingestion_ts|ingestion_date|source_system|api_status_code|
+----------+--------------------+--------------------+--------------+-------------+---------------+
|         1|{"abilities":[{"a...|2026-03-20 00:45:...|    2026-03-19|   pokeapi_co|            200|
|         2|{"abilities":[{"a...|2026-03-20 00:45:...|    2026-03-19|   pokeapi_co|            200|
|         3|{"abilities":[{"a...|2026-03-20 00:45:...|    2026-03-19|   pokeapi_co|            200|
+----------+--------------------+--------------------+--------------+-------------+---------------+
only showing top 3 rows


col_name,data_type,comment
pokemon_id,int,null
raw_json,string,null
ingestion_ts,timestamp,null
ingestion_date,string,null
source_system,string,null
api_status_code,int,null
# Partition Information,,
# col_name,data_type,comment
ingestion_date,string,null
,,


ingestion_date,total_records,valid_json,success_api
2026-03-19,151,151,151
